# Dan 1 — Tokeni, Temperatura i LLM Osnove

## Svrha ovog notebooka

U ovom notebooku ćemo:
1. Naučiti kako LLM modeli "vide" tekst (tokenizacija)
2. Uporediti tokenizaciju bosanskog i engleskog teksta
3. Izračunati cijenu API poziva
4. Eksperimentisati s temperaturom i vidjeti kako utječe na odgovore

**Prerequisites:**
- Python 3.11+ instaliran
- `.env` fajl kreiran (kopirajte `.env.example` u `.env`)
- DeepSeek API key ILI pokrenut Ollama server

**Pokretanje:** Izvršavajte ćelije redom koristeći `Shift+Enter`

In [ ]:
# =====================================================
# KORAK 1: Instalacija biblioteka
# Pokrenite ovu ćeliju JEDNOM na početku
# =====================================================

import subprocess
subprocess.run(["pip", "install", "tiktoken", "openai", "python-dotenv", "--quiet"], check=True)
print("Biblioteke instalirane!")

In [ ]:
# =====================================================
# KORAK 2: Uvoz biblioteka
# =====================================================

# tiktoken = OpenAI-jeva biblioteka za preciznu tokenizaciju
# Ovo je ISTI tokenizator koji koriste GPT-4, DeepSeek i drugi modeli
import tiktoken

# openai = SDK za komunikaciju s LLM API-jima
# Radi s OpenAI, DeepSeek i Ollama (OpenAI-kompatibilni API)
from openai import OpenAI

# os = za čitanje environment varijabli iz .env fajla
import os

# load_dotenv = učitava varijable iz .env fajla u os.environ
from dotenv import load_dotenv

# httpx = HTTP klijent za provjeru dostupnosti servisa
import httpx

# Učitavamo .env konfiguraciju
load_dotenv(dotenv_path="../.env")

print("Sve biblioteke učitane!")

## Tokenizacija s tiktoken

Browser demo (app.js) koristi **aproksimaciju** tokenizacije.
Ovdje koristimo **pravu tiktoken biblioteku** koja precizno broji tokene
identično kao OpenAI/DeepSeek API.

**Zašto je ovo važno?**
- Cijena API poziva se računa po broju tokena
- Kontekst prozor ima ograničen broj tokena
- Bosanski tekst troši VIŠE tokena od engleskog

In [ ]:
# =====================================================
# KORAK 3: Tokenizacija — kako LLM "vidi" tekst
# =====================================================

# Kreiramo enkoder za model cl100k_base (koristi ga GPT-4, DeepSeek)
# Enkoder je "rječnik" koji pretvara tekst u brojeve (tokene)
enc = tiktoken.get_encoding("cl100k_base")

# Primjer 1: Bosanski tekst
tekst_bs = "Sarajevo je glavni grad Bosne i Hercegovine."
tokeni_bs = enc.encode(tekst_bs)

print("=" * 60)
print("BOSANSKI TEKST")
print("=" * 60)
print(f"Tekst: {tekst_bs}")
print(f"Broj znakova: {len(tekst_bs)}")
print(f"Broj tokena: {len(tokeni_bs)}")
print(f"Token ID-ovi: {tokeni_bs}")
print()

# Dekodiramo svaki token da vidimo šta predstavlja
print("Tokeni pojedinačno:")
for i, token_id in enumerate(tokeni_bs):
    token_tekst = enc.decode([token_id])
    print(f"  Token {i+1}: ID={token_id:>6} → '{token_tekst}'")

print()

# Primjer 2: Isti sadržaj na engleskom
tekst_en = "Sarajevo is the capital city of Bosnia and Herzegovina."
tokeni_en = enc.encode(tekst_en)

print("=" * 60)
print("ENGLESKI TEKST (isti sadržaj)")
print("=" * 60)
print(f"Tekst: {tekst_en}")
print(f"Broj znakova: {len(tekst_en)}")
print(f"Broj tokena: {len(tokeni_en)}")
print()

# Poređenje
razlika = len(tokeni_bs) - len(tokeni_en)
print("=" * 60)
print("Poređenje")
print("=" * 60)
print(f"Bosanski: {len(tokeni_bs)} tokena za {len(tekst_bs)} znakova")
print(f"Engleski: {len(tokeni_en)} tokena za {len(tekst_en)} znakova")
print(f"Razlika: bosanski troši {abs(razlika)} tokena {'više' if razlika > 0 else 'manje'}")
print(f"Omjer znakova/token — BS: {len(tekst_bs)/len(tokeni_bs):.1f}, EN: {len(tekst_en)/len(tokeni_en):.1f}")

In [ ]:
# =====================================================
# KORAK 4: Kontekst prozor — koliko teksta stane?
# =====================================================

# Prosječna A4 stranica ima ~400 tokena
TOKENI_PO_STRANICI = 400

modeli = [
    ("DeepSeek V3", 128_000),
    ("GPT-4o", 128_000),
    ("GPT-4o-mini", 128_000),
    ("Llama 3.2 (lokalni)", 128_000),
    ("Claude 3.5 Sonnet", 200_000),
    ("Stariji modeli (GPT-3)", 4_096),
]

print("=" * 65)
print(f"{'Model':<25} {'Kontekst':>10} {'~Stranica A4':>15}")
print("=" * 65)

for naziv, kontekst in modeli:
    stranice = kontekst // TOKENI_PO_STRANICI
    print(f"{naziv:<25} {kontekst:>10,} tok   ~{stranice:>5} str.")

print()
print("Zaključak: Moderni modeli mogu 'pročitati' stotine stranica odjednom.")
print("Ali više teksta = veća cijena i sporiji odgovor.")

In [ ]:
# =====================================================
# KORAK 5: Cijena API poziva — koliko košta?
# =====================================================

# Cijene po 1 milion tokena (juni 2025)
cijene = [
    ("DeepSeek V3", 0.14, 0.28),
    ("GPT-4o-mini", 0.15, 0.60),
    ("GPT-4o", 2.50, 10.00),
    ("Claude Haiku", 0.25, 1.25),
    ("Llama (lokalni)", 0.00, 0.00),
]

# Simuliramo slanje 1000 tokena inputa i 500 tokena outputa
input_tokeni = 1000
output_tokeni = 500

print(f"Scenarij: {input_tokeni} input tokena + {output_tokeni} output tokena")
print(f"(Otprilike: 2 paragrafa pitanje → 1 paragraf odgovor)")
print()
print("=" * 65)
print(f"{'Model':<20} {'Input $/1M':>10} {'Output $/1M':>12} {'Cijena poziva':>15}")
print("=" * 65)

for naziv, cijena_in, cijena_out in cijene:
    ukupno = (input_tokeni / 1_000_000 * cijena_in) + (output_tokeni / 1_000_000 * cijena_out)
    print(f"{naziv:<20} ${cijena_in:>8.2f}   ${cijena_out:>9.2f}   ${ukupno:>12.6f}")

print()
print("Zaključak: DeepSeek je najjeftiniji cloud model.")
print("Za 1000 poziva dnevno, DeepSeek košta ~$0.28/dan.")
print("Lokalni model (Ollama) ne košta ništa, ali zahtijeva hardver.")

In [ ]:
# =====================================================
# KORAK 6: LLM klijent s fallbackom
# =====================================================

def get_llm_client():
    """
    Vraća LLM klijent prema dostupnosti.
    Prioritet: DeepSeek API → Ollama lokalno
    
    Ova funkcija automatski prebacuje na lokalni model
    ako cloud API nije dostupan ili API key nije postavljen.
    """
    deepseek_key = os.getenv("DEEPSEEK_API_KEY", "")
    
    # Pokušaj 1: DeepSeek API (cloud, bolji kvalitet)
    if deepseek_key and deepseek_key.startswith("sk-"):
        print("Koristim: DeepSeek API (cloud)")
        return OpenAI(
            api_key=deepseek_key,
            base_url="https://api.deepseek.com"
        ), "deepseek-chat"
    
    # Pokušaj 2: Ollama lokalno (fallback)
    ollama_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    try:
        httpx.get(f"{ollama_url}/api/tags", timeout=2.0)
        model = os.getenv("OLLAMA_MODEL_SMALL", "llama3.2:1b")
        print(f"Koristim: Ollama lokalno ({model})")
        return OpenAI(
            api_key="ollama",
            base_url=f"{ollama_url}/v1"
        ), model
    except Exception:
        raise RuntimeError(
            "Ni DeepSeek API ni Ollama nisu dostupni!\n"
            "Rješenje A: Dodajte DEEPSEEK_API_KEY u .env fajl\n"
            "Rješenje B: Pokrenite Ollama: docker-compose up ollama"
        )

# Kreiramo klijent
klijent, model_naziv = get_llm_client()

# Prvi API poziv — test
odgovor = klijent.chat.completions.create(
    model=model_naziv,
    messages=[
        {"role": "system", "content": "Odgovaraj kratko, na bosanskom jeziku."},
        {"role": "user", "content": "Objasni mi šta su tokeni u jednoj rečenici."}
    ],
    temperature=0.7,
    max_tokens=100
)

print(f"\nModel: {model_naziv}")
print(f"Odgovor: {odgovor.choices[0].message.content}")
if odgovor.usage:
    print(f"Tokeni — input: {odgovor.usage.prompt_tokens}, output: {odgovor.usage.completion_tokens}")

In [ ]:
# =====================================================
# KORAK 7: Temperature eksperiment
# =====================================================

prompt = "Objasni mi šta je vještačka inteligencija jednom rečenicom."

print("=" * 70)
print("EKSPERIMENT: Isti prompt na različitim temperaturama")
print(f"Prompt: \"{prompt}\"")
print("=" * 70)

# Temperature=0.0 — deterministički (uvijek isti odgovor)
print("\n--- Temperature = 0.0 (deterministički) ---")
print("Očekivanje: svih 5 odgovora bi trebali biti IDENTIČNI\n")

for i in range(5):
    odg = klijent.chat.completions.create(
        model=model_naziv,
        messages=[
            {"role": "system", "content": "Odgovaraj jednom rečenicom na bosanskom."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=100
    )
    print(f"  Pokušaj {i+1}: {odg.choices[0].message.content}")

# Temperature=0.7 — balansirano
print("\n--- Temperature = 0.7 (balansirano) ---")
print("Očekivanje: slični ali ne identični odgovori\n")

for i in range(3):
    odg = klijent.chat.completions.create(
        model=model_naziv,
        messages=[
            {"role": "system", "content": "Odgovaraj jednom rečenicom na bosanskom."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=100
    )
    print(f"  Pokušaj {i+1}: {odg.choices[0].message.content}")

# Temperature=1.5 — kreativno
print("\n--- Temperature = 1.5 (kreativno) ---")
print("Očekivanje: svaki odgovor je drugačiji, ponekad neočekivan\n")

for i in range(3):
    odg = klijent.chat.completions.create(
        model=model_naziv,
        messages=[
            {"role": "system", "content": "Odgovaraj jednom rečenicom na bosanskom."},
            {"role": "user", "content": prompt}
        ],
        temperature=1.5,
        max_tokens=100
    )
    print(f"  Pokušaj {i+1}: {odg.choices[0].message.content}")

print("\n" + "=" * 70)
print("ZAKLJUČAK:")
print("  temp=0.0 → isti odgovor svaki put (za SQL, JSON, egzaktne zadatke)")
print("  temp=0.7 → varijacije na temu (za chat, preporuke)")
print("  temp=1.5 → kreativni, nepredvidivi odgovori (za brainstorming)")
print("=" * 70)

## Zaključak Dana 1

Danas smo naučili:

1. **Tokeni** — osnovna jedinica teksta za LLM (~4 znaka na engleskom, ~2.5 na bosanskom)
2. **Kontekst prozor** — koliko teksta model može "vidjeti" odjednom (128K tokena = ~320 stranica)
3. **Cijena API poziva** — računa se po broju tokena (DeepSeek: $0.14/1M input)
4. **Temperatura** — kontroliše nasumičnost odgovora (0=determinističko, 1.5=kreativno)
5. **API fallback** — automatsko prebacivanje s clouda na lokalni model

---

## Vježbe za kod kuće

**Vježba 1.1:** Uzmite paragraf teksta iz internog dokumenta vaše institucije.  
Koliko tokena ima? Izračunajte koliko bi koštalo slanje tog dokumenta DeepSeek API-ju 1000 puta.

**Vježba 1.2:** Napišite prompt: "Objasni mi [vaš posao] jednom rečenicom."  
Pokrenite ga 5 puta s temperature=0 i 5 puta s temperature=1.5. Dokumentujte razlike.

**Vježba 1.3:** Koristite tiktoken da izračunate koliko strana PDF-a stane u kontekst prozor od 128,000 tokena.  
(Prosječna strana A4 ~400 tokena)

**Vježba 1.4:** Uzmite isti sadržaj na bosanskom i engleskom. Usporedite broj tokena. Koji jezik je "jeftiniji"?

**Vježba 1.5 (bonus):** Istražite: što se dogodi ako kontekst prozor postane pun?  
Napišite 2-3 rečenice odgovor u komentaru ispod.